#### Set Up

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import torch
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple
import os, pathlib, shutil, random
import string

In [3]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


Language Translator - English to Spanish

#### Load Dataset

In [4]:
zip_path = keras.utils.get_file(origin=("http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"), fname="spa-eng", extract=True, )
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"
text_path

WindowsPath('C:/Users/PRASHANTH N/.keras/datasets/spa-eng/spa-eng/spa.txt')

In [5]:
with open(text_path, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    spanish = "[start] " + spanish + " [end]"
    text_pairs.append((english, spanish))
text_pairs[22020]

('Tom lit a cigarette.', '[start] Tomás encendió un cigarrillo. [end]')

#### Data processing

In [6]:
random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples

train_pairs = text_pairs[:train_samples]
val_pairs = text_pairs[train_samples : train_samples + val_samples]
test_pairs = text_pairs[train_samples + val_samples :]

In [7]:
# Learning token vocabularies for English and Spanish text
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "").replace("]", "")
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", ""
    )
vocab_size = 15000
sequence_length = 20

In [8]:
english_tokenizer = keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

spanish_tokenizer = keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

In [9]:
# Tokenizing and preparing the translation data
batch_size = 64

def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    features = {"english": eng, "spanish": spa[:, :-1]}
    labels = spa[:, 1:]
    sample_weights = labels != 0
    return features, labels, sample_weights

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [10]:
inputs, targets, sample_weights = next(iter(train_ds))
print(inputs["english"].shape)
print(inputs["spanish"].shape)

(64, 20)
(64, 20)


#### RNN Seq2Seq Model

In [11]:
# Building a sequence-to-sequence encoder
embed_dim = 256
hidden_dim = 1024
source = keras.Input(shape=(None,), dtype="int32", name="english")
x = keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layer = keras.layers.GRU(hidden_dim)
rnn_layer = keras.layers.Bidirectional(rnn_layer, merge_mode="sum")
encoder_output = rnn_layer(x)


In [12]:
# Building a sequence-to-sequence decoder
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layer = keras.layers.GRU(hidden_dim, return_sequences=True)
x = rnn_layer(x, initial_state=encoder_output)
x = keras.layers.Dropout(0.5)(x)

target_predictions = keras.layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, target], target_predictions)
seq2seq_rnn.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spanish             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  3,840,000 │ english[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  3,840,000 │ spanish[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 1024)      │  7,876,608 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, None,      │  3,938,304 │ embedding_1[0][0… │
│                     │ 1024)             │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, None,      │          0 │ gru_1[0][0]       │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 15,375,000 │ dropout[0][0]     │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

#### Training

In [13]:
seq2seq_rnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", weighted_metrics=["accuracy"])
seq2seq_rnn.fit(train_ds, epochs=3, validation_data=val_ds)

Epoch 1/3


C:\Users\PRASHANTH N\PycharmProjects\MTechSem2\gpu\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:853: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n = torch._VF.gru(


1302/1302 ━━━━━━━━━━━━━━━━━━━━ 369s 282ms/step - accuracy: 0.3576 - loss: 3.6683 - val_accuracy: 0.4967 - val_loss: 2.4999
Epoch 2/3
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 409s 315ms/step - accuracy: 0.5358 - loss: 2.2603 - val_accuracy: 0.5896 - val_loss: 1.9089
Epoch 3/3
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 420s 322ms/step - accuracy: 0.6211 - loss: 1.6349 - val_accuracy: 0.6255 - val_loss: 1.6995


#### Inference

In [14]:
# Generating translations with a seq2seq RNN
spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]

for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
The picture on the wall is the one that Tom painted last summer.
[start] el cuadro de la caja de tom es el tesoro de la ciudad pasada [end]
-
If I happen to end up going abroad, I'd probably go for France.
[start] si yo llegue a la escuela si llegue a tokio [end]
-
Does this medicine work quickly?
[start] este trabajo funciona rápido [end]
-
Tom is a house painter.
[start] tom es un par de nervios [end]
-
I thought you might want this.
[start] pensé que podrías hacerlo esto [end]


In [15]:
input_sentence = "Hey, How are you doing?"
print(input_sentence)
print(generate_translation(input_sentence))

Hey, How are you doing?
[start] ¡hola qué estás haciendo [end]
